# Evaluate binding affinity and source-protein ligand recovery

Prepare the training table, download current IEDB evaluation records with `python src/download_iedb_eval_data.py`, and attach UniProt source sequences with `python src/download_source_proteins.py`. Install the external predictors as described in `README.md`, then run this notebook from the repository root.

The BA checkpoint supplies affinity predictions, and the EL checkpoint ranks source-protein windows. Run cells in order to filter the cohorts, compare predictions, calculate uncertainty, and save result tables. Cohort sizes depend on the downloaded data and predictor coverage. CPU, MPS, and CUDA inference are supported, subject to each external predictor's platform availability. Full FRANK inference can take hours.


In [ ]:
from pathlib import Path
import platform
import torch

# Run from the repository root after the README data/setup steps.
ROOT = Path.cwd().resolve()
if not (ROOT / "src" / "lamina.py").is_file():
    raise RuntimeError("Start this notebook from the LAMINA repository root")
DATA_DIR = ROOT / "Data" / "NetMHCpan"
TRAINING_DATA = DATA_DIR / "netmhcpan_training.jsonl.gz" # produced by src/prepare_netmhcpan_data.py
PSEUDOSEQUENCE_DATA = DATA_DIR / "NetMHCpan_train" / "pseudoseqs"
BA_DATA = DATA_DIR / "iedb_recent_ba.csv"
FRANK_DATA = DATA_DIR / "pearson_el_with_source.csv.gz"
BA_CHECKPOINT = ROOT / "final_models" / "lamina_ba.pt"
EL_CHECKPOINT = ROOT / "final_models" / "lamina_el.pt"

# Install the pinned upstream releases listed in README.md under external/.
EXTERNAL_DIR = ROOT / "external"
NETMHCPAN_HOME = EXTERNAL_DIR / "netMHCpan-4.2"
NETMHCPAN_PLATFORM_NAME = f"{platform.system()}_{platform.machine()}"
NETMHCPAN_BINARY = NETMHCPAN_HOME / NETMHCPAN_PLATFORM_NAME / "bin" / "netMHCpan-4.2"
NETMHCPAN_PLATFORM = NETMHCPAN_BINARY.parents[1]
NETMHCPAN_DATA = NETMHCPAN_PLATFORM / "data"
DEEPATTENTION_DIR = EXTERNAL_DIR / "DeepAttentionPan"
DEEPATTENTION_CODE = DEEPATTENTION_DIR / "codes"
ANTHEM_DIR = EXTERNAL_DIR / "ANTHEM"
ANTHEM_WEKA = ANTHEM_DIR / "source" / "weka-3-9-3" / "weka.jar"
TRANSPHLA_DIR = EXTERNAL_DIR / "TransPHLA-AOMP" / "TransPHLA-AOMP"

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

LAMINA_BATCH_SIZE = 1024
DEEPATTENTION_BATCH_SIZE = 128
TRANSPHLA_BATCH_SIZE = 256
SEED = 0
BOOTSTRAP_REPEATS = 2_000
FRANK_SIGN_FLIPS = 100_000
FRANK_BOOTSTRAPS = 10_000
# Result tables from this execution.
OUTPUT_DIR = ROOT / "artifacts" / "evals"


In [ ]:
import contextlib
import gzip
import html
import importlib
import importlib.util
import json
import math
import os
import re
import subprocess
import sys
import tempfile
from collections import defaultdict
from itertools import combinations

import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import pearsonr, spearmanr, wilcoxon
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)
from tqdm.auto import tqdm

from src.lamina import encode_batch, load_model

np.random.seed(SEED)
torch.manual_seed(SEED)

required_inputs = [TRAINING_DATA, PSEUDOSEQUENCE_DATA, BA_DATA, FRANK_DATA,
                   BA_CHECKPOINT, EL_CHECKPOINT]
missing_inputs = [str(path) for path in required_inputs if not path.is_file()]
if missing_inputs:
    raise FileNotFoundError("Complete the README setup steps; missing:\n" + "\n".join(missing_inputs))

ba_model = load_model(BA_CHECKPOINT, DEVICE)
el_model = load_model(EL_CHECKPOINT, DEVICE)

print(f"device: {DEVICE}")
print(f"LAMINA parameters: {sum(p.numel() for p in ba_model.parameters()):,}")


## Evaluation records

IEDB stores exact measurements either with an explicit equals sign or with no inequality prefix. Less-than and greater-than measurements are not quantitative point observations and are removed. The same exact peptide-pseudosequence exclusion is then applied to the BA and Pearson cohorts.


In [ ]:
CANONICAL_AA = set("ACDEFGHIKLMNPQRSTVWY")

def normalize_hla(value):
    """Return a fully typed four-digit HLA-A, -B, or -C name."""
    text = re.sub(r"\s+", "", str(value or "").upper())
    match = re.fullmatch(r"(?:HLA-)?([ABC])\*?(\d{2}):?(\d{2})", text)
    return (
        f"HLA-{match.group(1)}*{match.group(2)}:{match.group(3)}"
        if match else None
    )


def parse_exact_ic50(value):
    """Parse equality measurements and reject every inequality-qualified value."""
    text = html.unescape(str(value or "")).replace("\xa0", " ").strip()
    if text.startswith(("<", ">")):
        return None

    # IEDB uses both '= 123 nM' and the equivalent bare '123 nM'. Anchoring the
    # expression prevents numbers elsewhere in a malformed field from leaking in.
    match = re.fullmatch(
        r"(?:=\s*)?((?:\d+(?:\.\d*)?|\.\d+)(?:[eE][+-]?\d+)?)\s*(?:nM)?",
        text,
        flags=re.IGNORECASE,
    )
    if match is None:
        return None
    result = float(match.group(1))
    return result if np.isfinite(result) and result > 0 else None


def normalized_affinity(ic50_nm):
    """Apply the NetMHCpan target transform used to train the BA checkpoint."""
    ic50_nm = np.clip(float(ic50_nm), 1.0, 50_000.0)
    return 1.0 - math.log(ic50_nm) / math.log(50_000.0)


allele_to_pseudosequence = {}
with PSEUDOSEQUENCE_DATA.open() as handle:
    for line in handle:
        allele, pseudosequence = line.split()[:2]
        allele = normalize_hla(allele)
        if allele:
            allele_to_pseudosequence[allele] = pseudosequence.upper()


# Build the prospective quantitative BA cohort.
ba_rows = []
for row in pd.read_csv(BA_DATA).to_dict("records"):
    peptide = str(row.get("linear_sequence") or "").upper()
    allele = normalize_hla(row.get("mhc_allele_name"))
    ic50_nm = parse_exact_ic50(row.get("quantitative_measure"))

    if (
        8 <= len(peptide) <= 14
        and set(peptide) <= CANONICAL_AA
        and allele in allele_to_pseudosequence
        and ic50_nm is not None
    ):
        ba_rows.append({
            "record_id": f"ba:{int(row['elution_id'])}",
            "peptide": peptide,
            "allele": allele,
            "pseudoseq": allele_to_pseudosequence[allele],
            "affinity_nm": ic50_nm,
            "affinity_target": normalized_affinity(ic50_nm),
        })

ba = pd.DataFrame(ba_rows)
if ba.empty or ba.record_id.duplicated().any():
    raise ValueError("The parsed BA cohort is empty or contains duplicate record IDs")


# Load the source sequences attached by download_source_proteins.py.
frank = pd.read_csv(FRANK_DATA)
frank["peptide"] = frank.linear_sequence.fillna("").astype(str).str.upper()
frank["allele"] = frank.mhc_allele_name.map(normalize_hla)
frank["pseudoseq"] = frank.allele.map(allele_to_pseudosequence)
frank["accession"] = frank.accession.fillna("").astype(str)
frank["source_sequence"] = frank.source_sequence.fillna("").astype(str).str.upper()

frank = frank[
    (frank.status == "peptide_found_in_sequence") # quality control filter to ensure the actual presented peptide is present in the protein source sequence
    & frank.peptide.str.len().between(8, 14)
    & frank.peptide.map(lambda peptide: set(peptide) <= CANONICAL_AA)
    & frank.pseudoseq.notna()
    & frank.accession.ne("")
    & frank.source_sequence.ne("")
].drop_duplicates(["peptide", "allele", "accession"])

frank["record_id"] = frank.elution_id.map(lambda value: f"frank:{int(value)}")
frank["dataset"] = "Pearson et al."
frank = frank[[
    "record_id", "dataset", "peptide", "allele", "pseudoseq",
    "accession", "source_sequence",
]].reset_index(drop=True)


# Collect only overlaps relevant to these two evaluation cohorts
requested_true_pairs = (
    set(zip(ba.peptide, ba.pseudoseq))
    | set(zip(frank.peptide, frank.pseudoseq))
)
trained_true_pairs = set()

with gzip.open(TRAINING_DATA, "rt", encoding="utf-8") as handle:
    for line in tqdm(handle, desc="exact BA+EL train-pair scan", unit="row"):
        training_row = json.loads(line)
        if training_row["dataset"] not in {"ba", "el"}:
            continue
        peptide = training_row["peptide"].upper()
        for pseudosequence in training_row["pseudosequences"]:
            pair = (peptide, pseudosequence.upper())
            if pair in requested_true_pairs:
                trained_true_pairs.add(pair)


ba_before = len(ba)
frank_before = len(frank)
ba = ba[
    [(peptide, pseudoseq) not in trained_true_pairs
     for peptide, pseudoseq in zip(ba.peptide, ba.pseudoseq)]
].reset_index(drop=True)
frank = frank[
    [(peptide, pseudoseq) not in trained_true_pairs
     for peptide, pseudoseq in zip(frank.peptide, frank.pseudoseq)]
].reset_index(drop=True)

cohort_counts = pd.DataFrame([
    {
        "cohort": "prospective BA",
        "parsed": ba_before,
        "exact_train_pair_exclusions": ba_before - len(ba),
        "retained": len(ba),
        "alleles": ba.allele.nunique(),
    },
    {
        "cohort": "Pearson FRANK",
        "parsed": frank_before,
        "exact_train_pair_exclusions": frank_before - len(frank),
        "retained": len(frank),
        "alleles": frank.allele.nunique(),
    },
])
display(cohort_counts)
if ba.empty or frank.empty:
    raise ValueError("No evaluation records remain after filtering and training-pair exclusion")



## LAMINA inference


In [ ]:
@torch.inference_mode()
def score_lamina(frame, model, batch_size=LAMINA_BATCH_SIZE):
    """Score peptide-HLA rows with the selected BA or EL checkpoint."""
    batches = []

    for start in range(0, len(frame), batch_size):
        part = frame.iloc[start:start + batch_size]
        hla_ids, _ = encode_batch(part.pseudoseq.tolist(), 34)
        peptide_ids, peptide_lengths = encode_batch(part.peptide.tolist(), 14)

        hla_ids = hla_ids.to(DEVICE)
        peptide_ids = peptide_ids.to(DEVICE)
        peptide_lengths = peptide_lengths.to(DEVICE)

        # Use BF16 convolutions on CUDA with float32 accumulation.
        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.bfloat16,
            enabled=DEVICE.type == "cuda",
        ):
            logits, _ = model(hla_ids, peptide_ids, peptide_lengths)

        logits = logits.float().cpu().numpy()
        probabilities = 1.0 / (1.0 + np.exp(-logits))
        batches.append(pd.DataFrame({
            "record_id": part.record_id.to_numpy(),
            "classification_score": probabilities,
            "affinity_score": probabilities,
            "affinity_nm_pred": np.power(50_000.0, 1.0 - probabilities),
            "ranking_score": logits,
        }))

    return pd.concat(batches, ignore_index=True)


## External predictor adapters

These short adapters call the five released predictors without modifying or copying their repositories. NetMHCpan and ANTHEM require temporary input files; those files exist only for the duration of each call and are deleted automatically.


In [ ]:
def score_netmhcpan(frame, show_progress=True):
    """Return NetMHCpan EL, BA, and IC50 outputs aligned to the input rows."""
    rows = []

    with tempfile.TemporaryDirectory(prefix="netmhcpan-", dir="/tmp") as temporary:
        work = Path(temporary)

        for group_number, (pseudoseq, group) in enumerate(
            tqdm(
                frame.groupby("pseudoseq", sort=False),
                desc="NetMHCpan HLA",
                disable=not show_progress,
            )
        ):
            label = f"EVAL{group_number:05d}"
            pseudo_file = work / f"{label}.pseudo"
            name_file = work / f"{label}.names"
            peptide_file = work / f"{label}.peptides"

            # NetMHCpan accepts a synthetic allele name paired with the exact pseudosequence, so all tools see the same typed HLA representation
            pseudo_file.write_text(f"{label} {pseudoseq}\n")
            name_file.write_text(f"{label} {label}\n")
            peptide_file.write_text(
                "\n".join(dict.fromkeys(group.peptide)) + "\n"
            )

            # Every NetMHCpan path is explicit. In particular, the executable's
            # NETMHCpan environment-variable default is never used.
            command = [
                str(NETMHCPAN_BINARY),
                "-p", "-BA",
                "-a", label,
                "-rdir", str(NETMHCPAN_PLATFORM),
                "-syn", str(NETMHCPAN_DATA / "synlist_nocontext.bin"),
                "-version", str(NETMHCPAN_DATA / "version"),
                "-thrfmt", str(NETMHCPAN_DATA / "threshold" / "%s.thr.%s"),
                "-hlapseudo", str(pseudo_file),
                "-allname", str(name_file),
                "-f", str(peptide_file),
                "-tdir", str(work / f"run{group_number}"),
            ]
            completed = subprocess.run(
                command,
                text=True,
                capture_output=True,
                check=True,
            )

            # With -BA, columns 11, 13, and 15 are EL score, BA score, and nM.
            parsed = {}
            for line in completed.stdout.splitlines():
                fields = line.split()
                if len(fields) >= 16 and fields[0].isdigit():
                    parsed[fields[2]] = (
                        float(fields[11]),
                        float(fields[13]),
                        float(fields[15]),
                    )

            for row in group.itertuples():
                el_score, ba_score, affinity_nm = parsed.get(
                    row.peptide, (np.nan, np.nan, np.nan)
                )
                rows.append({
                    "record_id": row.record_id,
                    "classification_score": ba_score,
                    "affinity_score": ba_score,
                    "affinity_nm_pred": affinity_nm,
                    "ranking_score": el_score,
                })

    return pd.DataFrame(rows)


def score_mhcflurry(frame, predictor):
    """Use affinity for BA classification/regression and presentation for FRANK."""
    indexed = frame.copy()
    indexed["input_index"] = np.arange(len(indexed))

    alleles = list(dict.fromkeys(indexed.allele.astype(str)))
    sample_name = {
        allele: f"allele_{number}" for number, allele in enumerate(alleles)
    }

    scored = predictor.predict(
        peptides=indexed.peptide.astype(str).tolist(),
        alleles={
            sample_name[allele]: [allele]
            for allele in alleles
        },
        sample_names=[
            sample_name[allele] for allele in indexed.allele.astype(str)
        ],
        n_flanks=None,
        c_flanks=None,
        verbose=0,
        throw=False,
    ).rename(columns={"peptide_num": "input_index"})

    scored = indexed[["record_id", "input_index"]].merge(
        scored[["input_index", "affinity", "presentation_score"]],
        on="input_index",
        how="left",
    )

    affinity_nm = pd.to_numeric(scored.affinity, errors="coerce")
    # Invalid predictions remain missing; clipping infinity or a negative nM
    # value would otherwise turn an upstream failure into a confident score.
    affinity_nm = affinity_nm.where(np.isfinite(affinity_nm) & affinity_nm.gt(0))
    affinity_score = (
        1.0
        - np.log(affinity_nm.clip(lower=1.0, upper=50_000.0))
        / math.log(50_000.0)
    )

    presentation_score = pd.to_numeric(
        scored.presentation_score, errors="coerce"
    )
    return pd.DataFrame({
        "record_id": scored.record_id,
        "classification_score": affinity_score,
        "affinity_score": affinity_score,
        "affinity_nm_pred": affinity_nm,
        "ranking_score": presentation_score,
    })


In [ ]:
def score_deepattentionpan(frame):
    """Run the released twenty-model ensemble and average normalized affinity."""
    sys.path.insert(0, str(DEEPATTENTION_CODE))
    try:
        config_module = importlib.import_module("config_parser")
        model_module = importlib.import_module("model")
        encoding_module = importlib.import_module("seq_encoding")
    finally:
        sys.path.pop(0)

    def safe_attention(attention, features):
        identity = torch.eye(
            features.size(2),
            device=features.device,
            dtype=features.dtype,
        )
        identity = identity.unsqueeze(0).expand(features.size(0), -1, -1)
        weights = attention.sm(
            attention.fc(
                torch.cat([features, identity], 1)
                .permute(0, 2, 1)
                .contiguous()
            )
        )
        attended = (
            features.permute(0, 2, 1) * weights
        ).permute(0, 2, 1).contiguous()
        return attended, weights.reshape(weights.size(0), -1)

    model_module.Attention2.forward = safe_attention
    configuration = config_module.Config(
        str(DEEPATTENTION_CODE / "dup_0" / "config.json")
    )

    ensemble = []
    model_count = (
        int(configuration.base_model_count)
        * int(configuration.model_count)
    )
    for model_number in range(model_count):
        model = model_module.Model(configuration).to(DEVICE)
        model.load_state_dict(torch.load(
            configuration.model_save_path(model_number),
            map_location=DEVICE,
            weights_only=True,
        ))
        ensemble.append(model.eval())

    hla_sequences = {}
    sequence_file = DEEPATTENTION_DIR / "dataset" / "mhc_i_protein_seq2.txt"
    with sequence_file.open() as handle:
        for line_number, line in enumerate(handle):
            fields = line.split()
            if line_number and len(fields) == 2:
                hla_sequences[normalize_hla(fields[0])] = fields[1]

    affinity_scores = np.full(len(frame), np.nan)
    affinity_nm = np.full(len(frame), np.nan)
    supported = [
        index
        for index, row in enumerate(frame.itertuples())
        if row.allele in hla_sequences and 8 <= len(row.peptide) <= 15
    ]

    for start in tqdm(
        range(0, len(supported), DEEPATTENTION_BATCH_SIZE),
        desc="DeepAttentionPan",
    ):
        indices = supported[start:start + DEEPATTENTION_BATCH_SIZE]
        hla = torch.stack([
            encoding_module.blosum_encode(
                hla_sequences[frame.iloc[index].allele], 385
            )[0]
            for index in indices
        ]).to(DEVICE)
        peptide = torch.stack([
            encoding_module.blosum_encode(
                frame.iloc[index].peptide, 15
            )[0]
            for index in indices
        ]).to(DEVICE)

        with torch.inference_mode():
            predictions = torch.stack([
                model(
                    None, None, hla, None, None, None, peptide, None
                ).reshape(-1)
                for model in ensemble
            ])

        affinity_scores[indices] = (
            predictions.mean(0).float().cpu().numpy()
        )
        affinity_nm[indices] = (
            torch.pow(50_000.0, 1.0 - predictions)
            .mean(0)
            .float()
            .cpu()
            .numpy()
        )

    return pd.DataFrame({
        "record_id": frame.record_id,
        "classification_score": affinity_scores,
        "affinity_score": affinity_scores,
        "affinity_nm_pred": affinity_nm,
    })


def score_anthem(frame):
    """Recreate the five released Weka fold predictions for each HLA/length."""
    feature_names = ("AAF", "WLS", "PSSM", "PWM", "BSI62")
    matrix_names = ("AAF", "WLS", "IC50", "PWM", "BSI62")
    selection = json.loads(
        (ANTHEM_DIR / "source" / "20200511_dict_lenHLAnumselect.json")
        .read_text()
    )
    scores = np.full(len(frame), np.nan)

    groups = frame.groupby(["allele", frame.peptide.str.len()], sort=False)
    for (allele, length), group in tqdm(groups, desc="ANTHEM groups"):
        if str(length) not in selection or allele not in selection[str(length)]:
            continue

        compact_allele = (
            allele.removeprefix("HLA-").replace("*", "").replace(":", "")
        )
        matrices = [
            json.loads(
                (
                    ANTHEM_DIR / "feature_matrics" / name
                    / str(length) / f"{allele}.json"
                ).read_text()
            )
            for name in matrix_names
        ]

        features = []
        for peptide in group.peptide:
            aaf = sum(
                float(matrices[0][amino_acid][position])
                for position, amino_acid in enumerate(peptide)
            )
            wls = sum(
                float(matrices[1][amino_acid][position])
                for position, amino_acid in enumerate(peptide)
            )
            pssm = sum(
                float(matrices[2][amino_acid][position])
                for position, amino_acid in enumerate(peptide)
            ) / length
            pwm = sum(
                float(matrices[3][amino_acid][position])
                for position, amino_acid in enumerate(peptide)
            )
            bsi = sum(
                float(matrices[4][amino_acid][position])
                for position, amino_acid in enumerate(peptide)
            )
            features.append(
                (aaf, wls, 50_000 ** ((0.8 - pssm) / 1.6), pwm, bsi)
            )

        fold_scores = []
        with tempfile.TemporaryDirectory(prefix="anthem-") as temporary:
            temporary = Path(temporary)

            for fold in range(5):
                selected = [
                    int(value) - 1
                    for value in str(
                        selection[str(length)][allele][str(fold)]
                    )
                ]
                arff = temporary / f"fold{fold}.arff"

                with arff.open("w") as handle:
                    handle.write("@RELATION peptide\n\n")
                    for index in selected:
                        handle.write(
                            f"@ATTRIBUTE {feature_names[index]} REAL\n"
                        )
                    handle.write(
                        "@ATTRIBUTE class {1, -1}\n\n@DATA\n"
                    )
                    for values in features:
                        handle.write(
                            " ".join(str(values[index]) for index in selected)
                            + " ?\n"
                        )

                model_file = (
                    ANTHEM_DIR / "models" / str(length)
                    / f"{length}_{compact_allele}_{fold}"
                )
                command = [
                    "java", "-cp", str(ANTHEM_WEKA),
                    "weka.classifiers.meta.FilteredClassifier",
                    "-l", str(model_file),
                    "-T", str(arff),
                    "-classifications",
                    (
                        "weka.classifiers.evaluation.output.prediction."
                        "PlainText -distribution -decimals 12"
                    ),
                ]
                completed = subprocess.run(
                    command,
                    text=True,
                    capture_output=True,
                    check=True,
                )

                values = []
                in_predictions = False
                for line in completed.stdout.splitlines():
                    if "inst#" in line and "distribution" in line:
                        in_predictions = True
                        continue
                    fields = line.split()
                    if (
                        in_predictions
                        and fields
                        and fields[0].isdigit()
                        and "," in fields[-1]
                    ):
                        values.append(float(
                            fields[-1].replace("*", "").split(",")[0]
                        ))
                fold_scores.append(values)

        scores[group.index] = np.mean(fold_scores, axis=0)

    return pd.DataFrame({
        "record_id": frame.record_id,
        "classification_score": scores,
    })


@contextlib.contextmanager
def working_directory(path):
    """Temporarily satisfy an upstream module's relative file lookups."""
    previous = Path.cwd()
    os.chdir(path)
    try:
        yield
    finally:
        os.chdir(previous)


@torch.inference_mode()
def score_transphla(frame):
    """Run the released TransPHLA-AOMP pHLAIformer checkpoint."""
    import scipy

    # The release imports scipy.interp, removed in newer SciPy releases.
    if not hasattr(scipy, "interp"):
        scipy.interp = np.interp

    specification = importlib.util.spec_from_file_location(
        "_transphla_model",
        TRANSPHLA_DIR / "model.py",
    )
    module = importlib.util.module_from_spec(specification)
    with working_directory(TRANSPHLA_DIR):
        specification.loader.exec_module(module)

    module.use_cuda = DEVICE.type == "cuda"
    module.device = DEVICE
    model = module.Transformer().to(DEVICE)
    model.load_state_dict(torch.load(
        TRANSPHLA_DIR / "pHLAIformer.pkl",
        map_location=DEVICE,
        weights_only=True,
    ))
    model.eval()

    scores = np.full(len(frame), np.nan)
    supported = [
        index
        for index, row in enumerate(frame.itertuples())
        if 8 <= len(row.peptide) <= 14
    ]

    for start in tqdm(
        range(0, len(supported), TRANSPHLA_BATCH_SIZE),
        desc="TransPHLA-AOMP",
    ):
        indices = supported[start:start + TRANSPHLA_BATCH_SIZE]
        part = frame.iloc[indices]
        inputs = pd.DataFrame({
            "HLA": [f"EVAL{index}" for index in indices],
            "HLA_sequence": part.pseudoseq.to_numpy(),
            "peptide": part.peptide.to_numpy(),
        })

        with working_directory(TRANSPHLA_DIR):
            _, _, _, loader = module.read_predict_data(
                inputs, TRANSPHLA_BATCH_SIZE
            )

        values = []
        for peptide_ids, hla_ids in loader:
            logits, *_ = model(
                peptide_ids.to(DEVICE),
                hla_ids.to(DEVICE),
            )
            values.extend(
                torch.softmax(logits.float(), dim=1)[:, 1].cpu().tolist()
            )

        scores[indices] = values

    return pd.DataFrame({
        "record_id": frame.record_id,
        "classification_score": scores,
    })


## Score the BA cohort

All prediction tables are generated from the filtered rows in this run. Classification uses records with finite predictions from all six models. Correlations and errors use the intersection of the four models with affinity outputs. The LAMINA calibration plot includes all its finite affinity predictions.


In [ ]:
from mhcflurry import Class1PresentationPredictor

# Share the presentation predictor across BA and FRANK scoring.
mhcflurry_predictor = Class1PresentationPredictor.load()

model_tables = {
    "LAMINA": score_lamina(ba, ba_model),
    "NetMHCpan-4.2": score_netmhcpan(ba),
    "MHCflurry 2.2.1": score_mhcflurry(ba, mhcflurry_predictor),
    "DeepAttentionPan": score_deepattentionpan(ba),
    "ANTHEM": score_anthem(ba),
    "TransPHLA-AOMP": score_transphla(ba),
}

# Keep predictions in a simple long table. The original measured columns are
# joined once, so all following statistics can be read directly.
prediction_parts = []
for model_name, table in model_tables.items():
    table = table.copy()
    if table.record_id.duplicated().any():
        raise ValueError(f"{model_name} returned duplicate record IDs")
    if not set(table.record_id).issubset(set(ba.record_id)):
        raise ValueError(f"{model_name} returned unrequested record IDs")
    table["model"] = model_name
    prediction_parts.append(table)

ba_scores = pd.concat(prediction_parts, ignore_index=True).merge(
    ba,
    on="record_id",
    how="left",
    validate="many_to_one",
)

MODEL_ORDER = [
    "LAMINA",
    "NetMHCpan-4.2",
    "MHCflurry 2.2.1",
    "DeepAttentionPan",
    "ANTHEM",
    "TransPHLA-AOMP",
]
CONTINUOUS_MODELS = MODEL_ORDER[:4]

def common_finite_record_ids(frame, models, columns):
    """Intersect model coverage for this endpoint, including finite truth."""
    if frame.duplicated(["model", "record_id"]).any():
        raise ValueError("Duplicate model/record_id predictions")
    supported = []
    for model_name in models:
        rows = frame[frame.model.eq(model_name)]
        finite = np.isfinite(rows[list(columns)].to_numpy(float)).all(axis=1)
        supported.append(set(rows.loc[finite, "record_id"]))
    return set.intersection(*supported) if supported else set()


# Classification includes six tools; regression uses the four tools that
# supply quantitative affinity. Classifier-only coverage does not restrict
# the continuous-model comparisons.
common_ba_ids = common_finite_record_ids(
    ba_scores, MODEL_ORDER, ("affinity_nm", "classification_score")
)
common_affinity_ids = common_finite_record_ids(
    ba_scores, CONTINUOUS_MODELS, ("affinity_target", "affinity_score")
)
if not common_ba_ids or not common_affinity_ids:
    raise ValueError("Models have no common finite BA evaluation records")
print(
    f"All-model classification cohort: {len(common_ba_ids):,}; "
    f"continuous-affinity cohort: {len(common_affinity_ids):,}"
)


## Binding-affinity metrics and statistics

Classification compares the six-model intersection. Correlations, absolute errors, and paired Wilcoxon tests compare the four-model affinity intersection. Each confidence interval resamples paired records. Holm correction covers all continuous-model pairs.


In [ ]:
def record_bootstrap_ci(
    first,
    second,
    statistic,
    seed,
    require_two_classes=False,
):
    """Return a deterministic 95% percentile CI from record resampling."""
    first = np.asarray(first)
    second = np.asarray(second)
    rng = np.random.default_rng(seed)
    estimates = []

    # Resampling indices—not values independently—preserves the measured and
    # predicted value from the same experimental record.
    for _ in range(BOOTSTRAP_REPEATS):
        indices = rng.integers(0, len(first), len(first))
        sampled_first = first[indices]
        if require_two_classes and np.unique(sampled_first).size < 2:
            continue
        estimate = statistic(sampled_first, second[indices])
        if np.isfinite(estimate):
            estimates.append(float(estimate))

    return (
        np.percentile(estimates, [2.5, 97.5])
        if estimates else np.array([np.nan, np.nan])
    )


def holm_adjust(p_values):
    """Holm correction over finite tests without propagating missing P values."""
    values = np.asarray(p_values, dtype=float)
    adjusted = np.full(len(values), np.nan)
    finite = np.flatnonzero(np.isfinite(values))
    order = finite[np.argsort(values[finite])]
    running_maximum = 0.0
    for rank, original_index in enumerate(order):
        running_maximum = max(running_maximum, (len(order) - rank) * values[original_index])
        adjusted[original_index] = min(1.0, running_maximum)
    return adjusted


def significance_label(p_value):
    if not np.isfinite(p_value):
        return "NA"
    if p_value < 0.001:
        return "***"
    if p_value < 0.01:
        return "**"
    if p_value < 0.05:
        return "*"
    return "ns"


In [ ]:
common_scores = ba_scores[
    ba_scores.record_id.isin(common_ba_ids)
].copy()
continuous_scores = ba_scores[
    ba_scores.model.isin(CONTINUOUS_MODELS)
    & ba_scores.record_id.isin(common_affinity_ids)
].copy()

# AUROC and AUPRC are calculated at both conventional IC50 thresholds.
classification_rows = []
for threshold_number, threshold in enumerate((500, 50)):
    for model_number, model_name in enumerate(MODEL_ORDER):
        rows = common_scores[common_scores.model == model_name].sort_values("record_id")
        labels = (rows.affinity_nm <= threshold).to_numpy(int)
        scores = rows.classification_score.to_numpy(float)

        if np.unique(labels).size != 2:
            raise ValueError(f"{threshold} nM classification requires both binder classes")
        auroc = roc_auc_score(labels, scores)
        auprc = average_precision_score(labels, scores)
        auroc_ci = record_bootstrap_ci(
            labels,
            scores,
            roc_auc_score,
            SEED + 100 * threshold_number + 2 * model_number,
            require_two_classes=True,
        )
        auprc_ci = record_bootstrap_ci(
            labels,
            scores,
            average_precision_score,
            SEED + 100 * threshold_number + 2 * model_number + 1,
            require_two_classes=True,
        )

        classification_rows.append({
            "threshold_nm": threshold,
            "model": model_name,
            "n": len(rows),
            "binders": int(labels.sum()),
            "auroc": auroc,
            "auroc_ci_low": auroc_ci[0],
            "auroc_ci_high": auroc_ci[1],
            "evaluation_scope": "six-model common classification records",
            "auroc_ci": f"{auroc_ci[0]:.3f}-{auroc_ci[1]:.3f}",
            "auprc": auprc,
            "auprc_ci_low": auprc_ci[0],
            "auprc_ci_high": auprc_ci[1],
            "auprc_ci": f"{auprc_ci[0]:.3f}-{auprc_ci[1]:.3f}",
        })

classification_metrics = pd.DataFrame(classification_rows)
display(classification_metrics.round({"auroc": 3, "auprc": 3}))


# Correlations and absolute errors use only models that emit affinity.
correlation_rows = []
error_rows = []
for model_number, model_name in enumerate(CONTINUOUS_MODELS):
    rows = continuous_scores[continuous_scores.model == model_name].sort_values("record_id")
    measured = rows.affinity_target.to_numpy(float)
    predicted = rows.affinity_score.to_numpy(float)

    pearson_value = pearsonr(measured, predicted).statistic
    spearman_value = spearmanr(measured, predicted).statistic
    pearson_ci = record_bootstrap_ci(
        measured,
        predicted,
        lambda x, y: pearsonr(x, y).statistic,
        SEED + 1_000 + 2 * model_number,
    )
    spearman_ci = record_bootstrap_ci(
        measured,
        predicted,
        lambda x, y: spearmanr(x, y).statistic,
        SEED + 1_001 + 2 * model_number,
    )

    correlation_rows.extend([
        {
            "model": model_name,
            "evaluation_scope": "four-model common continuous-affinity records",
            "correlation": "Pearson r",
            "estimate": pearson_value,
            "ci_low": pearson_ci[0],
            "ci_high": pearson_ci[1],
            "n": len(rows),
        },
        {
            "model": model_name,
            "evaluation_scope": "four-model common continuous-affinity records",
            "correlation": "Spearman ρ",
            "estimate": spearman_value,
            "ci_low": spearman_ci[0],
            "ci_high": spearman_ci[1],
            "n": len(rows),
        },
    ])

    absolute_error = np.abs(predicted - measured)
    error_rows.extend({
        "record_id": record_id,
        "model": model_name,
        "absolute_error": error,
    } for record_id, error in zip(rows.record_id, absolute_error))

correlation_metrics = pd.DataFrame(correlation_rows)
absolute_errors = pd.DataFrame(error_rows)

error_summary = (
    absolute_errors.groupby("model", sort=False)
    .absolute_error.agg(
        n="count", mae="mean", sd_abs_error="std", median_abs_error="median",
        q25_abs_error=lambda values: values.quantile(.25),
        q75_abs_error=lambda values: values.quantile(.75),
    )
    .reset_index()
)
display(correlation_metrics.round(3))
display(error_summary.round(3))


# The signed-rank test is paired by record. It asks whether one model's error
# distribution is shifted relative to another, without assuming normal errors.
error_pivot = absolute_errors.pivot(
    index="record_id",
    columns="model",
    values="absolute_error",
)
wilcoxon_rows = []
for model_a, model_b in combinations(CONTINUOUS_MODELS, 2):
    paired = error_pivot[[model_a, model_b]].dropna()
    difference = paired[model_a] - paired[model_b]
    p_value = 1.0 if np.all(difference == 0) else wilcoxon(
        paired[model_a],
        paired[model_b],
        alternative="two-sided",
        zero_method="wilcox",
        method="auto",
    ).pvalue
    wilcoxon_rows.append({
        "model_a": model_a,
        "model_b": model_b,
        "n_paired": len(paired),
        "mean_error_a_minus_b": (
            paired[model_a] - paired[model_b]
        ).mean(),
        "raw_p": p_value,
    })

wilcoxon_tests = pd.DataFrame(wilcoxon_rows)
wilcoxon_tests["holm_p"] = holm_adjust(wilcoxon_tests.raw_p)
wilcoxon_tests["significance"] = (
    wilcoxon_tests.holm_p.map(significance_label)
)
display(wilcoxon_tests)


## Pearson source-protein recovery (FRANK)

Every source-protein position contributes one candidate for every length 8-14. Repeated peptide strings are retained in the positional list. We score each distinct sequence only once per HLA/source context, then restore positional multiplicity when counting candidates strictly above the true ligand.


In [ ]:
frank_contexts = []
candidate_peptides_by_pseudosequence = defaultdict(set)

# Records sharing a source protein and HLA share exactly the same candidate
# pool, but each experimentally observed ligand retains its own FRANK result.
for (accession, allele, pseudoseq), rows in frank.groupby(
    ["accession", "allele", "pseudoseq"],
    sort=True,
):
    source = rows.source_sequence.iloc[0]

    # Retain one candidate per source position, including repeated sequences.
    positional_candidates = [
        source[start:start + length]
        for length in range(8, 15)
        for start in range(len(source) - length + 1)
    ]

    frank_contexts.append({
        "accession": accession,
        "allele": allele,
        "pseudoseq": pseudoseq,
        "rows": rows.copy(),
        "candidates": positional_candidates,
    })
    candidate_peptides_by_pseudosequence[pseudoseq].update(
        positional_candidates
    )


# Make one second streaming pass to find training pairs among the candidates
# needed for the retained records. This is far smaller than holding every
# BA/EL pair in memory and is recomputed from scratch on every notebook run.
trained_candidate_pairs = set()
with gzip.open(TRAINING_DATA, "rt", encoding="utf-8") as handle:
    for line in tqdm(
        handle,
        desc="exact FRANK candidate train-pair scan",
        unit="row",
    ):
        training_row = json.loads(line)
        if training_row["dataset"] not in {"ba", "el"}:
            continue
        peptide = training_row["peptide"].upper()

        for pseudosequence in training_row["pseudosequences"]:
            pseudosequence = pseudosequence.upper()
            if (
                pseudosequence in candidate_peptides_by_pseudosequence
                and peptide
                in candidate_peptides_by_pseudosequence[pseudosequence]
            ):
                trained_candidate_pairs.add((peptide, pseudosequence))


# Exclude trained candidates; the true ligand passed the earlier overlap check.
candidate_count_by_record = []
for context in frank_contexts:
    true_peptides = set(context["rows"].peptide)
    pseudoseq = context["pseudoseq"]
    context["candidates"] = [
        peptide
        for peptide in context["candidates"]
        if (
            peptide in true_peptides
            or (peptide, pseudoseq) not in trained_candidate_pairs
        )
    ]
    candidate_count_by_record.extend(
        [len(context["candidates"])] * len(context["rows"])
    )

del candidate_peptides_by_pseudosequence
del trained_candidate_pairs

print(
    f"FRANK contexts: {len(frank_contexts):,}; "
    f"records: {len(frank):,}; "
    f"mean candidates: {np.mean(candidate_count_by_record):,.0f}; "
    f"median candidates: {np.median(candidate_count_by_record):,.0f}"
)


In [ ]:
FRANK_MODEL_ORDER = [
    "LAMINA",
    "NetMHCpan-4.2",
    "MHCflurry 2.2.1",
]
frank_result_rows = []

# Run one model at a time. Only unique sequence requests are sent to a model,
# while context['candidates'] below remains positional and may contain repeats.
for model_name in FRANK_MODEL_ORDER:
    for context in tqdm(
        frank_contexts,
        desc=f"FRANK {model_name}",
        unit="source/HLA",
    ):
        unique_candidates = list(dict.fromkeys(context["candidates"]))
        score_input = pd.DataFrame({
            "record_id": np.arange(len(unique_candidates)),
            "peptide": unique_candidates,
            "allele": context["allele"],
            "pseudoseq": context["pseudoseq"],
        })

        if model_name == "LAMINA":
            scored = score_lamina(score_input, el_model)
        elif model_name == "NetMHCpan-4.2":
            scored = score_netmhcpan(score_input, show_progress=False)
        else:
            scored = score_mhcflurry(
                score_input,
                mhcflurry_predictor,
            )

        # Align by record ID so an adapter may return predictions in any order.
        aligned = score_input[["record_id", "peptide"]].merge(
            scored[["record_id", "ranking_score"]],
            on="record_id", how="left", validate="one_to_one",
        )
        score_by_peptide = dict(zip(aligned.peptide, aligned.ranking_score))
        positional_scores = np.asarray([
            score_by_peptide[peptide]
            for peptide in context["candidates"]
        ], dtype=float)

        # A context with missing predictions is excluded from that model
        # before constructing the common finite-record intersection.
        if not np.isfinite(positional_scores).all():
            continue

        for record in context["rows"].itertuples(index=False):
            true_score = score_by_peptide[record.peptide]
            if not np.isfinite(true_score):
                continue

            # Strictly-higher comparisons give tied windows the same rank. The
            # denominator is the complete filtered positional candidate list.
            higher = int(np.count_nonzero(
                positional_scores > true_score
            ))
            total = len(positional_scores)
            frank_result_rows.append({
                "record_id": record.record_id,
                "model": model_name,
                "dataset": record.dataset,
                "peptide": record.peptide,
                "allele": record.allele,
                "accession": record.accession,
                "true_rank": higher + 1,
                "total_candidates": total,
                "frank": higher / total,
            })

frank_scores = pd.DataFrame(frank_result_rows)
if frank_scores.empty:
    raise ValueError("No finite FRANK predictions were returned")

frank_supported_ids = []
for model_name in FRANK_MODEL_ORDER:
    frank_supported_ids.append(set(
        frank_scores.loc[
            frank_scores.model == model_name,
            "record_id",
        ]
    ))
common_frank_ids = set.intersection(*frank_supported_ids)
frank_scores = frank_scores[
    frank_scores.record_id.isin(common_frank_ids)
].reset_index(drop=True)

if not common_frank_ids:
    raise ValueError("Models have no common finite FRANK records")
print(
    f"FRANK common records: {len(common_frank_ids):,}; "
    f"source/HLA clusters: "
    f"{frank_scores[['record_id', 'accession', 'allele']].drop_duplicates()[['accession', 'allele']].drop_duplicates().shape[0]:,}"
)


## FRANK statistics

Records from the same source protein and HLA share a candidate pool, so they are not independent. Each random sign is therefore assigned to a source-protein × HLA cluster, and each bootstrap draw resamples whole clusters.


In [ ]:
def frank_cluster_tests(frame):
    """Compare every FRANK model pair with cluster-aware inference."""
    pivot = frame.pivot(
        index="record_id",
        columns="model",
        values="frank",
    ).reindex(columns=FRANK_MODEL_ORDER).dropna()

    metadata = (
        frame.drop_duplicates("record_id")
        .set_index("record_id")
        .reindex(pivot.index)
    )
    clusters = (
        metadata.accession.astype(str)
        + "|"
        + metadata.allele.astype(str)
    )

    rows = []
    for pair_number, (model_a, model_b) in enumerate(
        combinations(FRANK_MODEL_ORDER, 2)
    ):
        difference = pivot[model_a] - pivot[model_b]

        # A cluster contributes the sum of its record-level paired differences.
        # Dividing a randomized sum by the fixed number of records gives the
        # null distribution of the mean paired difference.
        cluster_table = pd.DataFrame({
            "cluster": clusters,
            "difference": difference,
        }).groupby("cluster").difference.agg(["sum", "size"])

        cluster_sums = cluster_table["sum"].to_numpy(float)
        cluster_sizes = cluster_table["size"].to_numpy(int)
        observed = float(difference.mean())
        rng = np.random.default_rng(SEED + 40_000 + pair_number)

        extreme = 0
        completed = 0
        while completed < FRANK_SIGN_FLIPS:
            batch = min(5_000, FRANK_SIGN_FLIPS - completed)
            signs = (
                rng.integers(
                    0,
                    2,
                    size=(batch, len(cluster_sums)),
                )
                * 2
                - 1
            )
            randomized_means = (
                signs @ cluster_sums / len(difference)
            )
            extreme += np.count_nonzero(
                np.abs(randomized_means) >= abs(observed)
            )
            completed += batch

        # The +1 numerator and denominator correction prevents a reported zero
        # P value in a finite Monte Carlo randomization test.
        raw_p = (extreme + 1) / (FRANK_SIGN_FLIPS + 1)

        bootstrap_means = []
        completed = 0
        while completed < FRANK_BOOTSTRAPS:
            batch = min(1_000, FRANK_BOOTSTRAPS - completed)
            sampled_clusters = rng.integers(
                0,
                len(cluster_sums),
                size=(batch, len(cluster_sums)),
            )

            # Resampled clusters can contain different numbers of records, so
            # the denominator is resampled together with the difference sum.
            bootstrap_means.extend(
                cluster_sums[sampled_clusters].sum(axis=1)
                / cluster_sizes[sampled_clusters].sum(axis=1)
            )
            completed += batch

        ci_low, ci_high = np.percentile(
            bootstrap_means,
            [2.5, 97.5],
        )
        rows.append({
            "model_a": model_a,
            "model_b": model_b,
            "n_records": len(difference),
            "n_clusters": len(cluster_sums),
            "mean_difference": observed,
            "ci_low": ci_low,
            "ci_high": ci_high,
            "raw_p": raw_p,
        })

    tests = pd.DataFrame(rows)
    tests["holm_p"] = holm_adjust(tests.raw_p)
    tests["significance"] = tests.holm_p.map(significance_label)
    return tests


frank_summary = (
    frank_scores.groupby("model", sort=False)
    .frank.agg(["count", "mean", "median"])
    .reset_index()
)
frank_tests = frank_cluster_tests(frank_scores)
display(frank_summary.round(5))
display(frank_tests)


## Result tables

Save predictions, cohort counts, affinity metrics, error comparisons, and FRANK statistics as CSV files in `artifacts/evals/`.


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULT_STEM = f"{BA_CHECKPOINT.stem}__{EL_CHECKPOINT.stem}"
result_tables = {
    "cohort_counts": cohort_counts,
    "ba_predictions": ba_scores,
    "classification_metrics": classification_metrics,
    "affinity_correlations": correlation_metrics,
    "affinity_errors": absolute_errors,
    "affinity_error_summary": error_summary,
    "affinity_pairwise_holm": wilcoxon_tests,
    "frank_per_record": frank_scores,
    "frank_summary": frank_summary,
    "frank_pairwise_holm": frank_tests,
}
for name, table in result_tables.items():
    table.to_csv(OUTPUT_DIR / f"{RESULT_STEM}_{name}.csv", index=False)
print(f"Saved {len(result_tables)} result tables to {OUTPUT_DIR}")
